In [4]:
import sys
import os
import json
import torch
import shap
import numpy as np
import pandas as pd

# Point to your project modules
sys.path.append(os.path.abspath('../'))
from src.models.sql_models import SQLCNN
from src.utils.data_helpers import encode_query

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open('../data/processed/sql_word2idx.json', 'r') as f:
    word2idx = json.load(f)

idx2word = {idx: word for word, idx in word2idx.items()}

model = SQLCNN(vocab_size=len(word2idx)).to(device)
# Ensure the path matches where your .pth file is actually saved
model.load_state_dict(torch.load('SQL_CNN_best.pth', map_location=device, weights_only=True))
model.eval()

# 1. Create a wrapper function that SHAP can call blindly
def predict_fn(queries_numpy):
    # Convert incoming numpy arrays to PyTorch long tensors
    tensor = torch.tensor(queries_numpy, dtype=torch.long).to(device)
    with torch.no_grad():
        raw_logits = model(tensor).squeeze(1)
        preds = torch.sigmoid(raw_logits).cpu().numpy()
    return preds

# 2. Prepare Background Data 
df = pd.read_csv('../data/raw/Modified_SQL_Dataset.csv').dropna()
normal_queries = df[df['Label'] == 0]['Query'].astype(str).values[:25] # Kept small for speed

background_data = np.array([encode_query(q, word2idx) for q in normal_queries])

# 3. Initialize the Model-Agnostic Kernel Explainer
explainer = shap.KernelExplainer(predict_fn, background_data)
print("SHAP KernelExplainer initialized successfully.")

SHAP KernelExplainer initialized successfully.


In [6]:
# 1. Define and encode a malicious test query
test_query = "SELECT * FROM users WHERE id = 1 OR 1=1 --"
encoded_test = encode_query(test_query, word2idx)
test_numpy = np.array([encoded_test])

# 2. Get the prediction
prediction = predict_fn(test_numpy)[0]
print(f"Query: {test_query}")
print(f"Malicious Probability: {prediction * 100:.2f}%\n")

# 3. Calculate SHAP values (passing the NumPy array, not a Tensor)
print("Calculating SHAP values (this may take a few seconds)...")
shap_values = explainer.shap_values(test_numpy)

# 4. Map the scores back to the actual words
tokens = [idx2word.get(idx, '<PAD>') for idx in encoded_test]

explanation = []
for word, score in zip(tokens, shap_values[0]):
    if word != '<PAD>':
        explanation.append((word, score))

# Sort by the most impactful words
explanation.sort(key=lambda x: x[1], reverse=True)

print("\nTop Keywords Triggering the 'Malicious' Flag:")
for word, score in explanation[:5]:
    print(f"Word: '{word:<10}' | SHAP Impact Score: +{score:.4f}")

Query: SELECT * FROM users WHERE id = 1 OR 1=1 --
Malicious Probability: 100.00%

Calculating SHAP values (this may take a few seconds)...


100%|██████████| 1/1 [00:23<00:00, 23.75s/it]


Top Keywords Triggering the 'Malicious' Flag:
Word: '=         ' | SHAP Impact Score: +0.2136
Word: '=         ' | SHAP Impact Score: +0.2135
Word: '-         ' | SHAP Impact Score: +0.2057
Word: 'or        ' | SHAP Impact Score: +0.1913
Word: '-         ' | SHAP Impact Score: +0.1706


In [7]:
import numpy as np

# Assuming 'background_data' is still in your notebook memory from earlier
np.save('../data/processed/shap_background.npy', background_data)
print("Background data saved for Streamlit!")

Background data saved for Streamlit!
